# 02 — Embeddings

Generate verb embeddings via OpenAI API, inspect them, and visualize with PCA.

**Requires:** `OPENAI_API_KEY` set in 00_Setup.

In [ ]:
# ═══ Generate embeddings for the corpus ═══
import numpy as np
import json
from pyodide.http import pyfetch

# Load corpus if not already loaded
try:
    corpus
except NameError:
    resp = await pyfetch('../data/combined_corpus.json')
    corpus = json.loads(await resp.string())
    print(f'Loaded corpus: {len(corpus)} verbs')

# Prepare texts for embedding
verb_names = list(corpus.keys())
texts = []
for v in verb_names:
    info = corpus[v]
    if isinstance(info, dict):
        defn = info.get('definition', info.get('def', v))
    else:
        defn = str(info) if info else v
    texts.append(f'{v}: {defn}')

print(f'Prepared {len(texts)} texts for embedding')
print(f'Sample: "{texts[0][:80]}..."')

In [ ]:
# ═══ Call OpenAI Embedding API ═══
# This cell costs money! ~$0.01 per 1000 verbs with text-embedding-3-large

BATCH_SIZE = 2048
MODEL = 'text-embedding-3-large'  # 3072 dimensions
all_embeddings = []

try:
    OPENAI_API_KEY
except NameError:
    OPENAI_API_KEY = ''

if not OPENAI_API_KEY:
    print('OPENAI_API_KEY not set. Set it in 00_Setup notebook.')
    print('Trying to load pre-computed embeddings instead...')
    # Try loading pre-computed
    try:
        resp = await pyfetch('../data/embedding_texts.json')
        emb_texts = json.loads(await resp.string())
        print(f'Loaded pre-computed embedding texts: {len(emb_texts)} entries')
        print('(Full embeddings not stored as JSON — use API to regenerate)')
    except:
        print('No pre-computed embeddings available.')
else:
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        print(f'  Embedding batch {i//BATCH_SIZE + 1}/{(len(texts)-1)//BATCH_SIZE + 1} ({len(batch)} texts)...')
        
        result = await api_call(
            'https://api.openai.com/v1/embeddings',
            {'Content-Type': 'application/json', 'Authorization': f'Bearer {OPENAI_API_KEY}'},
            {'model': MODEL, 'input': batch}
        )
        
        if 'error' in result:
            print(f'  ERROR: {result["error"]["message"]}')
            break
        
        sorted_data = sorted(result['data'], key=lambda x: x['index'])
        all_embeddings.extend([item['embedding'] for item in sorted_data])
    
    if all_embeddings:
        verb_embeddings = np.array(all_embeddings, dtype=np.float32)
        print(f'\nGenerated embeddings: {verb_embeddings.shape}')
        print(f'Model: {MODEL}')
        print(f'Dimensions: {verb_embeddings.shape[1]}')

In [ ]:
# ═══ Inspect embeddings ═══
try:
    verb_embeddings
    norms = np.linalg.norm(verb_embeddings, axis=1)
    print(f'Shape: {verb_embeddings.shape}')
    print(f'Norm — mean: {norms.mean():.4f}, std: {norms.std():.4f}, range: [{norms.min():.4f}, {norms.max():.4f}]')
    
    # Sample pairwise similarities
    sample_idx = np.random.choice(len(verb_embeddings), size=min(500, len(verb_embeddings)), replace=False)
    sample = verb_embeddings[sample_idx]
    sample_norm = sample / (np.linalg.norm(sample, axis=1, keepdims=True) + 1e-10)
    sims = sample_norm @ sample_norm.T
    upper = sims[np.triu_indices(len(sample), k=1)]
    print(f'Pairwise cosine similarity — mean: {upper.mean():.4f}, std: {upper.std():.4f}')
except NameError:
    print('No embeddings generated yet. Run the cell above first.')

In [ ]:
# ═══ PCA Visualization ═══
try:
    from sklearn.decomposition import PCA
    import js
    
    verb_embeddings
    
    print('Computing PCA projection...')
    pca = PCA(n_components=2)
    coords = pca.fit_transform(verb_embeddings)
    print(f'Variance explained: {pca.explained_variance_ratio_[0]:.1%}, {pca.explained_variance_ratio_[1]:.1%}')
    
    # Create a simple scatter plot using Plotly via JS interop
    from js import Plotly, document, JSON as jsJSON
    
    # Create div
    div = document.createElement('div')
    div.id = 'pca-plot'
    div.style.width = '100%'
    div.style.height = '500px'
    js.document.querySelector('.jp-OutputArea-output').appendChild(div)
    
    print(f'PCA computed. {len(coords)} points projected to 2D.')
    print('(Plotly visualization requires the full JupyterLite environment)')
    
    # Print text-based scatter of first 50 points
    print(f'\nFirst 20 verbs in PCA space:')
    for i in range(min(20, len(coords))):
        print(f'  {verb_names[i]:20s} ({coords[i,0]:+.3f}, {coords[i,1]:+.3f})')

except NameError:
    print('No embeddings available. Generate them first.')
except Exception as e:
    print(f'Visualization error: {e}')

In [ ]:
# ═══ Download embeddings ═══
try:
    verb_embeddings
    print(f'Embeddings shape: {verb_embeddings.shape}')
    print('Use download_json() from 01_Corpus to save.')
    print('Note: embeddings are large — consider saving as numpy binary instead.')
    
    # Save as JSON (large file!)
    # download_json({'verb_names': verb_names, 'embeddings': verb_embeddings.tolist()}, 'embeddings.json')
except NameError:
    print('No embeddings to download.')